In [14]:
from pathlib import Path
import importlib
%matplotlib qt

import matplotlib.pyplot as plt
import phase_bin_tools as pbt

pbt = importlib.reload(pbt)

bandpass_filter = pbt.bandpass_filter
extract_point_waveform = pbt.extract_point_waveform
infer_points_per_frame_from_filename = pbt.infer_points_per_frame_from_filename
infer_scan_rate_hz_from_filename = pbt.infer_scan_rate_hz_from_filename
list_phase_bin_files = pbt.list_phase_bin_files
plot_point_waveform = pbt.plot_point_waveform
plot_space_time = pbt.plot_space_time
read_multi_channel_phase_bin = pbt.read_multi_channel_phase_bin

plt.rcParams['figure.dpi'] = 120


In [27]:
# Cell 1: configure folder, file range, and plotting parameters
folder_path = Path(r"F:\\das5")
file_pattern = "0000*.bin"
points_per_frame = None
scan_rate_hz = None
point_index = 100

# Use 1-based inclusive indices in the matched-file list.
# Example: 1~5 means reading the first 5 matched files and concatenating them.
# Set both to None to use all matched files.
file_index_start = 1
file_index_end = 5

space_time_frame_slice = slice(None)
space_time_point_slice = slice(None)
space_time_vmin = -0.1
space_time_vmax = 0.1
space_time_cmap = 'jet'

enable_bandpass = True
bandpass_low_hz = 400.0
bandpass_high_hz = 999.0
bandpass_order = 4

show_point_waveform = True

print('folder_path:', folder_path)
print('file_pattern:', file_pattern)
print('file_index_range:', f'{file_index_start}~{file_index_end}' if file_index_start is not None or file_index_end is not None else 'all')
print('point_index:', point_index)
print('enable_bandpass:', enable_bandpass)


folder_path: F:\das5
file_pattern: 0000*.bin
file_index_range: 1~5
point_index: 100
enable_bandpass: True


In [28]:
# Cell 2: list matched files and select a serial range
all_matched_files = list_phase_bin_files(folder_path, pattern=file_pattern)

if file_index_start is None and file_index_end is None:
    selected_start = 1
    selected_end = len(all_matched_files)
else:
    selected_start = 1 if file_index_start is None else file_index_start
    selected_end = len(all_matched_files) if file_index_end is None else file_index_end

if selected_start <= 0 or selected_end <= 0:
    raise ValueError('file_index_start and file_index_end must be positive integers.')
if selected_start > selected_end:
    raise ValueError('file_index_start must be <= file_index_end.')
if selected_end > len(all_matched_files):
    raise ValueError(
        f'Requested file index range {selected_start}~{selected_end}, but only {len(all_matched_files)} files matched.'
    )

selected_files = all_matched_files[selected_start - 1:selected_end]

print('all matched file count:', len(all_matched_files))
print('selected file range:', f'{selected_start}~{selected_end}')
print('selected file count:', len(selected_files))
for file_index, path in enumerate(selected_files, start=selected_start):
    print(f'  [{file_index}]', path.name)


all matched file count: 10
selected file range: 1~5
selected file count: 5
  [1] 0000544-eDAS-2000Hz-0130pt-20260323T114035.252.bin
  [2] 0000545-eDAS-2000Hz-0130pt-20260323T114135.262.bin
  [3] 0000546-eDAS-2000Hz-0130pt-20260323T114235.251.bin
  [4] 0000547-eDAS-2000Hz-0130pt-20260323T114335.250.bin
  [5] 0000548-eDAS-2000Hz-0130pt-20260323T114435.362.bin


In [29]:
# Cell 3: read and concatenate the selected files as phase in radians
if points_per_frame is None:
    points_per_frame = infer_points_per_frame_from_filename(selected_files[0])
if points_per_frame is None:
    raise ValueError('Unable to infer points_per_frame from file name; set points_per_frame manually.')

if scan_rate_hz is None:
    scan_rate_hz = infer_scan_rate_hz_from_filename(selected_files[0])
if scan_rate_hz is None:
    raise ValueError('Unable to infer scan_rate_hz from file name; set scan_rate_hz manually.')

frame_data_phase = read_multi_channel_phase_bin(
    selected_files,
    points_per_frame=points_per_frame,
)

print('points_per_frame:', points_per_frame)
print('scan_rate_hz:', scan_rate_hz)
print('phase shape:', frame_data_phase.shape)
print('phase dtype:', frame_data_phase.dtype)
print('phase min/max (rad):', frame_data_phase.min(), frame_data_phase.max())


points_per_frame: 130
scan_rate_hz: 2000.0
phase shape: (600000, 130)
phase dtype: float64
phase min/max (rad): -38.34743025866558 36.20726999945872


In [30]:
# Cell 4: apply band-pass filtering along the time axis before plotting
if enable_bandpass:
    filtered_phase = bandpass_filter(
        frame_data_phase,
        sample_rate_hz=scan_rate_hz,
        lowcut_hz=bandpass_low_hz,
        highcut_hz=bandpass_high_hz,
        order=bandpass_order,
        axis=0,
    )
else:
    filtered_phase = frame_data_phase

print('bandpass enabled:', enable_bandpass)
if enable_bandpass:
    print('bandpass low/high/order:', bandpass_low_hz, bandpass_high_hz, bandpass_order)
    print('filtered min/max (rad):', filtered_phase.min(), filtered_phase.max())


bandpass enabled: True
bandpass low/high/order: 400.0 999.0 4
filtered min/max (rad): -5.41061259666485 5.21067459711806


In [31]:
# Cell 5: optionally compare one point waveform before and after filtering
if show_point_waveform:
    raw_waveform = extract_point_waveform(frame_data_phase, point_index)
    filtered_waveform = extract_point_waveform(filtered_phase, point_index)

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    plot_point_waveform(
        raw_waveform,
        sample_rate_hz=scan_rate_hz,
        title=f'Original Point Waveform (point={point_index})',
        ylabel='Phase (rad)',
        ax=axes[0],
    )
    plot_point_waveform(
        filtered_waveform,
        sample_rate_hz=scan_rate_hz,
        title=f'Bandpassed Point Waveform (point={point_index})',
        ylabel='Phase (rad)',
        ax=axes[1],
    )
    plt.tight_layout()
    plt.show()


In [32]:
# Cell 6: compare original and filtered space-time plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# plot_space_time(
#     frame_data_phase,
#     frame_slice=space_time_frame_slice,
#     point_slice=space_time_point_slice,
#     sample_rate_hz=scan_rate_hz,
#     cmap=space_time_cmap,
#     title='Original Space-Time Plot',
#     colorbar_label='Phase (rad)',
#     vmin=space_time_vmin,
#     vmax=space_time_vmax,
#     ax=axes[0],
# )


space_time_vmin = -0.05
space_time_vmax = 0.05

filtered_title = 'Bandpassed Space-Time Plot'
if enable_bandpass:
    filtered_title = f'Bandpassed Space-Time Plot ({bandpass_low_hz:g}-{bandpass_high_hz:g} Hz)'

plot_space_time(
    filtered_phase,
    frame_slice=space_time_frame_slice,
    point_slice=space_time_point_slice,
    sample_rate_hz=scan_rate_hz,
    cmap=space_time_cmap,
    title=filtered_title,
    colorbar_label='Phase (rad)',
    vmin=space_time_vmin,
    vmax=space_time_vmax,
    ax=axes[1],
)

plt.tight_layout()
plt.show()
